In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir('/content/drive/MyDrive/ABSA-BERT-pair')
print("Thư mục hiện tại:", os.getcwd())

Thư mục hiện tại: /content/drive/MyDrive/ABSA-BERT-pair


## 1. Import Required Libraries

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Load and Explore SemEval Dataset

In [ ]:
# Configuration
DATA_DIR = './data/semeval2014/bert-pair' # Changed path to be relative to the current working directory
TASK_NAME = 'semeval_QA_M'  # QA Multi-class classification task
LABELS = ['positive', 'neutral', 'negative', 'conflict', 'none']  # Multi-class labels

# Load training data
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_QA_M.csv'), sep='\t', header=None)
train_df.columns = ['id', 'label', 'question', 'sentence']

# Load test data
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_QA_M.csv'), sep='\t', header=None)
test_df.columns = ['id', 'label', 'question', 'sentence']

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print("\nSample training data:")
print(train_df.head())

Training samples: 15220
Test samples: 4000

Sample training data:
     id     label                                    question  \
0  3121      none      what do you think of the price of it ?   
1  3121      none  what do you think of the anecdotes of it ?   
2  3121      none       what do you think of the food of it ?   
3  3121      none   what do you think of the ambience of it ?   
4  3121  negative    what do you think of the service of it ?   

                               sentence  
0  But the staff was so horrible to us.  
1  But the staff was so horrible to us.  
2  But the staff was so horrible to us.  
3  But the staff was so horrible to us.  
4  But the staff was so horrible to us.  


## 3. Tokenize Data with Transformers

In [ ]:
# Load tokenizer and model
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_data(df, tokenizer, max_length=128):
    """Tokenize question and sentence pairs"""
    encodings = tokenizer(
        df['question'].tolist(),
        df['sentence'].tolist(),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    # Convert string labels to indices based on LABELS
    label_to_idx = {label: idx for idx, label in enumerate(LABELS)}
    labels = torch.tensor([label_to_idx[label] for label in df['label'].tolist()], dtype=torch.long)

    return encodings, labels

print("Tokenizing training data...")
train_encodings, train_labels = tokenize_data(train_df, tokenizer)

print("Tokenizing test data...")
test_encodings, test_labels = tokenize_data(test_df, tokenizer)

print(f"\nTrain encodings shape: {train_encodings['input_ids'].shape}")
print(f"Train labels shape: {train_labels.shape}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing training data...
Tokenizing test data...

Train encodings shape: torch.Size([15220, 102])
Train labels shape: torch.Size([15220])


## 4. Create PyTorch DataLoader

In [ ]:
# Create custom dataset class
class BertDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

# Create datasets
train_dataset = BertDataset(train_encodings, train_labels)
test_dataset = BertDataset(test_encodings, test_labels)

# Create DataLoaders
BATCH_SIZE = 16
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")

Training batches: 952
Test batches: 250


## 5. Load Pretrained BERT Model

In [ ]:
# Load pretrained BERT model for binary classification
NUM_LABELS = len(LABELS)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

# Move model to device
model.to(device)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of labels: {NUM_LABELS}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: bert-base-uncased
Number of labels: 5
Total parameters: 109,486,085


## 6. Set Up Training Loop

In [ ]:
# Training parameters
EPOCHS = 1
LEARNING_RATE = 2e-5
WARMUP_STEPS = 0

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_dataloader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

# Define training and evaluation functions
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader, desc="Training"):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    """Evaluate model"""
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            labels = batch['labels'].cpu().numpy()

            predictions.extend(preds)
            true_labels.extend(labels)

    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(true_labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(true_labels, predictions, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, predictions, true_labels

print("Training setup complete!")

Training setup complete!


## 7. Train the Model

In [ ]:
# Training loop
train_losses = []
all_results = []

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")

    # Train
    train_loss = train_epoch(model, train_dataloader, optimizer, scheduler, device)
    train_losses.append(train_loss)
    print(f"Train Loss: {train_loss:.4f}")

    # Save model checkpoint
    checkpoint_dir = './bert_qa_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.save_pretrained(f'{checkpoint_dir}/epoch_{epoch+1}')
    tokenizer.save_pretrained(f'{checkpoint_dir}/epoch_{epoch+1}')
    print(f"Model saved to {checkpoint_dir}/epoch_{epoch+1}")

print("\nTraining complete!")


Epoch 1/1


Training:  15%|█▌        | 146/952 [40:09<3:35:25, 16.04s/it]

## 8. Evaluate on Test Set

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

accuracy, precision, recall, f1, predictions, true_labels = evaluate(model, test_dataloader, device)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

# Save results to file
results = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'predictions': predictions,
    'true_labels': true_labels
}

import json
with open('./bert_qa_results.json', 'w') as f:
    json.dump({k: v if k != 'predictions' and k != 'true_labels' else str(v)
               for k, v in results.items()}, f, indent=4)
print("\nResults saved to bert_qa_results.json")

## 9. Generate Predictions and Visualize Results

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(true_labels, predictions)

print("\nConfusion Matrix:")
print("Labels:", LABELS)
# Print header with label names
header = "\t" + "\t".join(LABELS)
print(header)
# Print confusion matrix rows with label names
for i, label in enumerate(LABELS):
    row = str(label) + "\t" + "\t".join(str(cm[i, j]) for j in range(len(LABELS)))
    print(row)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion Matrix - BERT QA Multi-class Classification')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

# Set ticks
ax.set_xticks(range(len(LABELS)))
ax.set_yticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=45)
ax.set_yticklabels(LABELS)

# Add text annotations
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        text = ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, marker='o', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

print("\n" + "="*60)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("="*60)

# BERT for Question Answering with Transformers
Simplified implementation using Hugging Face Transformers library for SemEval-2014 dataset